# 04.1 Time-Series Model Training

Entrenamiento reproducible de `PriceSequenceGRU` sobre `data/processed_ts/`.

**Objetivo**
- cargar el dataset secuencial ya validado
- mantener un split temporal 80/20 coherente con el baseline
- entrenar una GRU pequeña y estable como segundo modelo
- guardar checkpoints y curvas para comparar después con `MarketValueNet`

**Artefactos esperados**
- `data/models/ts_gru/best_ts_gru_model.pt`
- `data/models/ts_gru/last_ts_gru_model.pt`
- `data/models/ts_gru/training_history.json`
- `data/models/ts_gru/run_config.json`

## 1. Setup

La primera celda carga config, paths y dependencias. Si `data/processed_ts/` no existe, primero hay que correr `python -m src.features.ts_pipeline`.

In [ ]:
import sys
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve, auc

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.model.ts_dataset import TimeSeriesMarketDataset, create_ts_dataloaders
from src.model.ts_architecture import PriceSequenceGRU
from src.model.ts_train import train_ts_model
from src.model.ts_evaluate import evaluate_ts_model, print_ts_evaluation

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
PROCESSED_TS = ROOT / cfg['ts_data']['processed_dir']
MODELS_DIR = ROOT / cfg['ts_training']['save_dir']
FIGURES = ROOT / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
assert PROCESSED_TS.exists(), f'Dataset TS no encontrado: {PROCESSED_TS}'


## 2. Inspección del dataset de entrada

Antes de entrenar conviene revisar metadata y confirmar que el dataset cargado coincide con el notebook 03.1.

In [ ]:
dataset = TimeSeriesMarketDataset.from_numpy_dir(str(PROCESSED_TS))
n = len(dataset)
print('Metadata dataset TS:')
print(json.dumps(dataset.metadata, indent=2))
print(f'Número de muestras: {n:,}')
print(f'Shape secuencias:   {tuple(dataset.sequences.shape)}')
print(f'Buy rate:           {100 * dataset.labels.mean().item():.1f}%')


## 3. Split temporal de train y validación

No mezclamos mercados futuros en entrenamiento. La validación se toma del tramo más reciente del dataset ordenado por fecha de resolución.

In [ ]:
VAL_SPLIT = cfg['training']['val_split']
BATCH_SIZE = cfg['ts_training']['batch_size']
SEED = cfg['ts_training']['seed']

train_loader, val_loader = create_ts_dataloaders(
    dataset,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    temporal_split=True,
    use_weighted_sampler=True,
    seed=SEED,
)

n_val = int(n * VAL_SPLIT)
n_train = n - n_val
print(f'Train: {n_train:,} | Val: {n_val:,}')

ts = dataset.timestamps
sorted_ts = np.sort(ts)
cutoff_ts = sorted_ts[n_train]
import datetime
cutoff_date = datetime.datetime.fromtimestamp(cutoff_ts, tz=datetime.timezone.utc)
print(f'Corte temporal TS: {cutoff_date.strftime("%Y-%m-%d")}')

fig, ax = plt.subplots(figsize=(10, 3))
dates = [datetime.datetime.fromtimestamp(t, tz=datetime.timezone.utc) for t in sorted_ts]
ax.scatter(dates[:n_train], [0]*n_train, alpha=0.3, s=5, color=PALETTE[0], label=f'Train ({n_train:,})')
ax.scatter(dates[n_train:], [0]*n_val, alpha=0.3, s=5, color=PALETTE[1], label=f'Val ({n_val:,})')
ax.axvline(cutoff_date, color='black', linestyle='--', lw=1.5, label='Corte')
ax.set_title('Split temporal del dataset TS')
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'ts_train_temporal_split.png', bbox_inches='tight')
plt.show()


## 4. Arquitectura del modelo

`PriceSequenceGRU` es un clasificador secuencial puro:
- entrada por timestep: `price_yes`, `delta_price`, `delta_time_scaled`
- encoder: GRU unidireccional
- cabeza final: `64 -> 32 -> 1` con `sigmoid`

La idea es que aprenda forma, dirección y ritmo de la trayectoria del precio antes del snapshot.

In [ ]:
model = PriceSequenceGRU(
    input_dim=dataset.sequences.shape[-1],
    hidden_dim=cfg['ts_model']['hidden_dim'],
    num_layers=cfg['ts_model']['num_layers'],
    dropout=cfg['ts_model']['dropout'],
    task='classification',
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Parámetros totales:     {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}')
print(f'Dispositivo: {"cuda" if torch.cuda.is_available() else "cpu"}')


## 5. Entrenamiento reproducible

Este paso fija seed, usa `AdamW`, early stopping y guarda el mejor checkpoint por `val_auc`. El baseline actual no se toca: este entrenamiento vive en paralelo.

In [ ]:
history = train_ts_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=cfg['ts_training']['epochs'],
    lr=cfg['ts_training']['learning_rate'],
    patience=cfg['ts_training']['patience'],
    seed=cfg['ts_training']['seed'],
    save_dir=str(MODELS_DIR),
    run_config={
        'source': '04_1_ts_model_training.ipynb',
        'dataset_metadata': dataset.metadata,
        'ts_model': cfg['ts_model'],
        'ts_training': cfg['ts_training'],
    },
)


## 6. Curvas de aprendizaje

Aquí buscamos tres cosas:
- que la loss de validación no diverja
- que `val_auc` supere claramente el azar
- que `val_pr_auc` no colapse si hay desbalance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], color=PALETTE[0], label='Train loss')
axes[0].plot(epochs_range, history['val_loss'], color=PALETTE[1], label='Val loss')
axes[0].set_title('Loss por época')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(epochs_range, history['val_auc'], color=PALETTE[2], linewidth=2, label='Val AUC-ROC')
axes[1].axhline(0.5, color='gray', linestyle=':', lw=1)
axes[1].set_title('Val AUC-ROC')
axes[1].set_xlabel('Época')
axes[1].legend()

axes[2].plot(epochs_range, history['val_pr_auc'], color=PALETTE[3], linewidth=2, label='Val PR-AUC')
axes[2].set_title('Val PR-AUC')
axes[2].set_xlabel('Época')
axes[2].legend()

plt.suptitle('Curvas de aprendizaje — PriceSequenceGRU', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'ts_training_curves.png', bbox_inches='tight')
plt.show()


## 7. Evaluación del mejor checkpoint

Cargamos `best_ts_gru_model.pt` y medimos desempeño en validación temporal. Esta es la referencia que luego se usará en `05_2_model_comparison.ipynb`.

In [ ]:
best_model = PriceSequenceGRU(
    input_dim=dataset.sequences.shape[-1],
    hidden_dim=cfg['ts_model']['hidden_dim'],
    num_layers=cfg['ts_model']['num_layers'],
    dropout=cfg['ts_model']['dropout'],
    task='classification',
)
best_model.load_state_dict(torch.load(MODELS_DIR / 'best_ts_gru_model.pt', map_location='cpu', weights_only=True))

results = evaluate_ts_model(best_model, val_loader)
print_ts_evaluation(results)


## 8. Diagnóstico visual del modelo

Las gráficas finales ayudan a responder:
- si el umbral `0.5` separa algo útil
- si la ROC realmente está por encima del azar
- si la confusión entre `Buy` y `No Buy` es razonable

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
cm = results['confusion_matrix']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Buy', 'Buy'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix')

ax = axes[1]
scores = results['scores']
labels_arr = results['labels']
for val, label, color in [(1, 'Buy', PALETTE[2]), (0, 'No Buy', PALETTE[3])]:
    sub = scores[labels_arr == val]
    if len(sub) > 0:
        ax.hist(sub, bins=30, alpha=0.6, color=color, density=True, label=label)
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold 0.5')
ax.set_title('Distribución de scores')
ax.legend(fontsize=9)

ax = axes[2]
if len(np.unique(labels_arr)) >= 2:
    fpr, tpr, _ = roc_curve(labels_arr, scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=PALETTE[0], lw=2, label=f'AUC = {roc_auc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Validación con una sola clase\nROC no informativa', ha='center', va='center')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')

plt.suptitle('Evaluación en validación — PriceSequenceGRU', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'ts_training_evaluation.png', bbox_inches='tight')
plt.show()


## 9. Siguiente paso

Si el checkpoint quedó guardado y las métricas no se ven degeneradas, ya puedes pasar a:
- `05_1_ts_live_scoring.ipynb` para usar el modelo en activos
- `05_2_model_comparison.ipynb` para compararlo contra `MarketValueNet`